<a href="https://colab.research.google.com/github/Autosnight/COMP90042_2026/blob/andrew-branch/embedding_to_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#installing libraries
%pip install transformers

In [14]:
# 1. Force update core libraries and install the specific missing module
%pip install --upgrade transformers sentence-transformers
%pip install configuration_eurobert
print("Libraries updated and configuration_eurobert installed. PLEASE RESTART THE SESSION (Runtime > Restart session) before continuing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2


Libraries updated. PLEASE RESTART THE SESSION (Runtime > Restart session) before continuing.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression #using log reg due to quick training and inference, good baseline measure
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

# Loading data
/content/evidence.json
and /content/COMP90042_2026/data/train-claims.json
convert claims into table with 2 colums: claim text and labels
convert evidence into list of strings

In [8]:
import json
import pandas as pd

# Load claims data
with open('/content/COMP90042_2026/data/train-claims.json', 'r') as f:
    claims_data = json.load(f)

# Convert claims to DataFrame
# Based on format: {"id": {"claim_text": "...", "claim_label": "...", "evidences": [...]}}
claims_df = pd.DataFrame.from_dict(claims_data, orient='index')

# Rename or select columns to match the expected 'claim_text' and 'label' variables
# using 'claim_label' as provided in your format snippet
claims_df = claims_df[['claim_text', 'claim_label']]
claims_df.rename(columns={'claim_label': 'label'}, inplace=True)

# Load evidence data
with open('/content/evidence.json', 'r') as f:
    evidence_data = json.load(f)

# Convert evidence into a list of strings
evidence_list = list(evidence_data.values())

display(claims_df.head())
print(f"Loaded {len(claims_df)} claims and {len(evidence_list)} evidence items.")

,claim_text,label
claim-1937,Not only is there no scientific evidence that ...,DISPUTED
claim-126,El Niño drove record highs in global temperatu...,REFUTES
claim-2510,"In 1946, PDO switched to a cool phase.",SUPPORTS
claim-2021,Weather Channel co-founder John Coleman provid...,DISPUTED
claim-2449,"""January 2008 capped a 12 month period of glob...",NOT_ENOUGH_INFO


Loaded 1228 claims and 1208827 evidence items.


# Loading frozen models + embeddings
1 cell: load distilbert and jina v5. jina v5 retrieval nano model.
different cell: use sentence transformer to convert all claims and evidence into jina embeddings (jina.encode). claims should be query embeddings, evidence should be document embeddings


In [2]:
# 1. Load Jina v5 retrieval nano model
import torch
from sentence_transformers import SentenceTransformer

try:
    # trust_remote_code=True is essential for Jina v5's custom 'eurobert' architecture
    jina_model = SentenceTransformer(
        "jinaai/jina-embeddings-v5-text-nano",
        trust_remote_code=True,
        model_kwargs={"dtype": torch.float32}
    )
    print("Jina v5 Model loaded successfully!")
except Exception as e:
    print(f"Error: {e}")
    print("\n--- TROUBLESHOOTING ---")
    print("If you still see an ImportError, ensure you have clicked 'Restart session' in the Runtime menu.")

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- configuration_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/424M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Error: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

--- TROUBLESHOOTING ---
If you still see an ImportError, ensure you have clicked 'Restart session' in the Runtime menu.


### Loading Jina v5 with Custom Config
If you have just restarted your session, run the cell below to load the model. The `trust_remote_code=True` parameter is essential for downloading the custom `eurobert` architecture.

In [ ]:
# Generate embeddings using Jina v5 specific parameters
claim_texts = claims_df['claim_text'].tolist()

print("Encoding claims as queries...")
claim_embeddings = jina_model.encode(
    sentences=claim_texts,
    task="retrieval",
    prompt_name="query"
)

print("Encoding evidence as documents (this may take a moment)... index: 1.2M items")
# Note: Encoding 1.2M items will take significant time/RAM.
# Consider using a subset for testing if needed.
evidence_embeddings = jina_model.encode(
    sentences=evidence_list,
    task="retrieval",
    prompt_name="document"
)

In [10]:
#retrieve class and weights
model_class, pretrained_weights = (ppb.DistilBertModel, 'distilbert-base-uncased')
#load model
frozen_bert = model_class.from_pretrained(pretrained_weights)

# retrieval
use jina similarity function with jina embeddings to get 2 most similar texts from documents/evidence for each claim. add them to a dictionary with key = claim (string) and value = evidences (string)

In [ ]:
import numpy as np

claim_evidence_map = {}

# Calculate similarity using jina_model's similarity function
# jina_model.similarity returns a matrix of cosine similarities
similarities = jina_model.similarity(claim_embeddings, evidence_embeddings)

# Convert to numpy if it's a tensor for easier indexing
if hasattr(similarities, 'numpy'):
    similarities = similarities.numpy()

for i, claim in enumerate(claim_texts):
    # Get indices of top 2 highest similarity scores
    top_2_indices = np.argsort(similarities[i])[-2:][::-1]
    retrieved_evidence = [evidence_list[idx] for idx in top_2_indices]
    claim_evidence_map[claim] = retrieved_evidence

print(f"Mapped {len(claim_evidence_map)} claims to their top 2 evidence pieces using Jina similarity.")

# concatination:
concatinate these strings together and add sep tokens between and then encode them for distilbert with padding  

In [ ]:
tokenizer = ppb.DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

concatenated_inputs = []
for claim, evidences in claim_evidence_map.items():
    # Concatenate: [CLS] Claim [SEP] Evidence1 [SEP] Evidence2 [SEP]
    text = f"{claim} [SEP] {evidences[0]} [SEP] {evidences[1]}"
    concatenated_inputs.append(text)

tokenized_features = tokenizer(
    concatenated_inputs,
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt'
)

# classifier embeddings
use distilbert to create cls embeddings of concatinated claim-evidence-evidence strings

In [ ]:
frozen_bert.eval()
all_features = []

with torch.no_grad():
    # Processing in small batches to avoid OOM
    batch_size = 8
    for i in range(0, len(concatenated_inputs), batch_size):
        batch_ids = tokenized_features['input_ids'][i:i+batch_size]
        batch_mask = tokenized_features['attention_mask'][i:i+batch_size]

        outputs = frozen_bert(batch_ids, attention_mask=batch_mask)
        # Extract [CLS] token embeddings (first token)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_features.append(cls_embeddings)

features = np.concatenate(all_features, axis=0)
labels = claims_df['label'].values
print(f"Feature matrix shape: {features.shape}")

#

# Training classification model

---
## input:

features and labels,

Output: accuracy scores in train and test set


In [ ]:
# Splitting the data
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

# Initialize and train the Logistic Regression model
completed_classifier = LogisticRegression(max_iter=1000)
completed_classifier.fit(train_features, train_labels)

# Calculate accuracy scores
train_score = completed_classifier.score(train_features, train_labels)
test_score = completed_classifier.score(test_features, test_labels)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")

In [ ]:
import json

# Get predictions from the trained classifier
predictions = completed_classifier.predict(features)

# We need the original evidence IDs for the output format
evidence_keys = list(evidence_data.keys())
output_results = {}

# Iterate through the original claims_data to maintain ID consistency
for i, (claim_id, claim_info) in enumerate(claims_data.items()):
    # Re-retrieve the top 2 indices for this specific claim to get the evidence IDs
    top_2_indices = np.argsort(similarities[i])[-2:][::-1]
    retrieved_evidence_ids = [evidence_keys[idx] for idx in top_2_indices]

    output_results[claim_id] = {
        "claim_text": claim_info['claim_text'],
        "claim_label": predictions[i],
        "evidences": retrieved_evidence_ids
    }

# Save to a JSON file
with open('predictions.json', 'w') as f:
    json.dump(output_results, f, indent=2)

print("Results formatted and saved to predictions.json")
# Display a sample of the formatted output
print(json.dumps(dict(list(output_results.items())[:2]), indent=2))